In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

In [ ]:
folder = "/content/drive/MyDrive/YELP"
os.listdir(folder)[:50]

['yelp_academic_dataset_business.json',
 'yelp_academic_dataset_user.json',
 'yelp_academic_dataset_checkin.json',
 'yelp_academic_dataset_tip.json',
 'yelp_academic_dataset_review.json',
 'Dataset_User_Agreement.pdf',
 'yelp_dataset',
 'restaurant_yelp.csv',
 'review_yelp.csv',
 'review_yelp_750k.csv']

In [ ]:
business_path = f"{folder}/yelp_academic_dataset_business.json"
review_path = f"{folder}/yelp_academic_dataset_review.json"

In [ ]:
#build Restaurant file

business_df = pd.read_json(business_path, lines=True)

print("Business: ", business_df.shape)

Business:  (150346, 14)


In [ ]:
restaurant_df =  business_df[
    business_df["categories"].fillna("").str.contains("Restaurants", case=False)
].copy()

In [ ]:

restaurant_df = restaurant_df[restaurant_df["review_count"] > 0]

print("All Businesses: ", business_df.shape)
print("Only Restaurants: ", restaurant_df.shape)

All Businesses:  (150346, 14)
Only Restaurants:  (52268, 14)


In [ ]:
#save to drive
restaurant_csv = f"{folder}/restaurant_yelp.csv"
restaurant_df.to_csv(restaurant_csv, index=False)
print("Saved:", restaurant_csv)

Saved: /content/drive/MyDrive/YELP/restaurant_yelp.csv


In [ ]:
restaurant_df = pd.read_csv(restaurant_csv)

In [ ]:
restaurant_id = set(restaurant_df["business_id"])
len(restaurant_id)

52268

In [ ]:
#load reviews
rev_chunks = []
kept = 0

for chunk in pd.read_json(review_path, lines=True, chunksize=100000):
  chunk = chunk[chunk["business_id"].isin(restaurant_id)]
  rev_chunks.append(chunk)
  kept += len(chunk)
  print("Kept: ", kept)

rev_rest = pd.concat(rev_chunks, ignore_index=True)
print("Reviews: ", rev_rest.shape)

Kept:  72124
Kept:  144424
Kept:  215621
Kept:  283029
Kept:  346454
Kept:  408261
Kept:  468131
Kept:  540923
Kept:  615448
Kept:  689501
Kept:  760016
Kept:  825773
Kept:  888802
Kept:  949191
Kept:  1020912
Kept:  1094854
Kept:  1168863
Kept:  1239914
Kept:  1306306
Kept:  1367454
Kept:  1428397
Kept:  1498435
Kept:  1571277
Kept:  1643372
Kept:  1711090
Kept:  1774018
Kept:  1834911
Kept:  1894634
Kept:  1964773
Kept:  2036484
Kept:  2108492
Kept:  2175552
Kept:  2237444
Kept:  2295596
Kept:  2355690
Kept:  2428624
Kept:  2501609
Kept:  2574542
Kept:  2641185
Kept:  2703921
Kept:  2764670
Kept:  2827974
Kept:  2902021
Kept:  2975887
Kept:  3048650
Kept:  3116105
Kept:  3180197
Kept:  3240090
Kept:  3303086
Kept:  3376781
Kept:  3450951
Kept:  3523644
Kept:  3592225
Kept:  3657548
Kept:  3717567
Kept:  3779489
Kept:  3853180
Kept:  3927022
Kept:  3999515
Kept:  4067019
Kept:  4133206
Kept:  4193227
Kept:  4254955
Kept:  4328479
Kept:  4402123
Kept:  4474727
Kept:  4543675
Kept:  460

In [ ]:
review_df = rev_rest.copy()
print("Total Reviews: ", len(review_df))
review_df.head()

Total Reviews:  4724471


,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11
1,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30
2,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03
3,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,1,0,1,Cute interior and owner (?) gave us tour of up...,2017-01-14 20:54:15
4,JrIxlS1TzJ-iCu79ul40cQ,eUta8W_HdHMXPzLBBZhL1A,04UD14gamNjLY0IDYVhHJg,1,1,2,1,I am a long term frequent customer of this est...,2015-09-23 23:10:31


In [ ]:
review_df.columns

Index(['review_id', 'user_id', 'business_id', 'stars', 'useful', 'funny',
       'cool', 'text', 'date'],
      dtype='object')

In [ ]:
review_df['date'] = pd.to_datetime(review_df['date'], errors='coerce')
review_df = review_df.dropna(subset=['date'])

review_df['year'] = review_df['date'].dt.year
review_df['month'] = review_df['date'].dt.month
review_df['day'] = review_df['date'].dt.day

review_df[['date', 'year', 'month']].head()

,date,year,month
0,2018-07-07 22:09:11,2018,7
1,2014-02-05 20:30:30,2014,2
2,2015-01-04 00:01:03,2015,1
3,2017-01-14 20:54:15,2017,1
4,2015-09-23 23:10:31,2015,9


In [ ]:
#clean the text

import re

def cleantext(s):
  s = str(s).lower()
  s = re.sub(r"http\S+", "", s)
  s = re.sub(r"\s+", " ", s)
  s = re.sub(r"[^a-z0-9\s'!?.,]", "", s)
  return s.strip()

review_df["text_cl"] = review_df["text"].apply(cleantext)
review_df["text_cl"].head()

,text_cl
0,"if you decide to eat here, just be aware it is..."
1,family diner. had the buffet. eclectic assortm...
2,"wow! yummy, different, delicious. our favorite..."
3,cute interior and owner ? gave us tour of upco...
4,i am a long term frequent customer of this est...


In [ ]:
review_sv = f"{folder}/review_yelp.csv"
review_df.to_csv(review_sv, index=False)
print("Saved: ", review_sv)

Saved:  /content/drive/MyDrive/YELP/review_yelp.csv


---
# NEW SECTION - RUN THIS ONLY

In [ ]:
from google.colab import drive
import os
import pandas as pd
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,  mean_squared_error
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [ ]:
folder = "/content/drive/MyDrive/YELP"
os.listdir(folder)[:50]

['yelp_academic_dataset_business.json',
 'yelp_academic_dataset_user.json',
 'yelp_academic_dataset_checkin.json',
 'yelp_academic_dataset_tip.json',
 'yelp_academic_dataset_review.json',
 'Dataset_User_Agreement.pdf',
 'yelp_dataset',
 'restaurant_yelp.csv',
 'review_yelp.csv',
 'review_yelp_750k.csv']

In [ ]:
restaurant_df = pd.read_csv(f"{folder}/restaurant_yelp.csv")
restaurant_id = set(restaurant_df["business_id"])
print("restaurants:", len(restaurant_id))

restaurants: 52268


In [ ]:
in_path = f"{folder}/review_yelp.csv"
out_path = f"{folder}/review_yelp_750k.csv"

In [ ]:
tar_row = 750_000
chunksize = 200_000

kept = 0
first = True

for chunk in pd.read_csv(in_path, chunksize=chunksize):
  chunk = chunk[chunk["business_id"].isin(restaurant_id)]

  rem = tar_row - kept
  if len(chunk) > rem:
    chunk = chunk.head(rem)

  chunk.to_csv(out_path, mode="a", index=False, header=first)
  first = False

  kept += len(chunk)
  print("Kept: ", kept)

  if kept >= tar_row:
    break

print("Done, saved to: ", out_path)

Kept:  200000
Kept:  400000
Kept:  600000
Kept:  750000
Done, saved to:  /content/drive/MyDrive/YELP/review_yelp_200k.csv


In [ ]:
#Try and merge two files

folder = "/content/drive/MyDrive/YELP"

review_df = pd.read_csv(f"{folder}/review_yelp_750k.csv")

In [ ]:
restaurant_df = pd.read_csv(f"{folder}/restaurant_yelp.csv")

print("Reviews: ", review_df.shape)
print("Restaurants: ", restaurant_df.shape)

Reviews:  (750000, 13)
Restaurants:  (52268, 14)


In [ ]:
restaurant_df.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
1,CF33F8-E6oudUQ46HnavjQ,Sonic Drive-In,615 S Main St,Ashland City,TN,37015,36.269593,-87.058943,2.0,6,1,"{'BusinessParking': 'None', 'BusinessAcceptsCr...","Burgers, Fast Food, Sandwiches, Food, Ice Crea...","{'Monday': '0:0-0:0', 'Tuesday': '6:0-22:0', '..."
2,k0hlBqXX-Bt0vf1op7Jr1w,Tsevi's Pub And Grill,8025 Mackenzie Rd,Affton,MO,63123,38.565165,-90.321087,3.0,19,0,"{'Caters': 'True', 'Alcohol': ""u'full_bar'"", '...","Pubs, Restaurants, Italian, Bars, American (Tr...",NaN
3,bBDDEgkFA1Otx9Lfe7BZUQ,Sonic Drive-In,2312 Dickerson Pike,Nashville,TN,37207,36.208102,-86.768170,1.5,10,1,"{'RestaurantsAttire': ""'casual'"", 'Restaurants...","Ice Cream & Frozen Yogurt, Fast Food, Burgers,...","{'Monday': '0:0-0:0', 'Tuesday': '6:0-21:0', '..."
4,eEOYSgkmpB90uNA7lDOMRA,Vietnamese Food Truck,NaN,Tampa Bay,FL,33602,27.955269,-82.456320,4.0,10,1,"{'Alcohol': ""'none'"", 'OutdoorSeating': 'None'...","Vietnamese, Food, Restaurants, Food Trucks","{'Monday': '11:0-14:0', 'Tuesday': '11:0-14:0'..."


In [ ]:
review_df.head()

,review_id,user_id,business_id,stars,useful,funny,cool,text,date,year,month,day,text_cl
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11,2018,7,7,"if you decide to eat here, just be aware it is..."
1,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30,2014,2,5,family diner. had the buffet. eclectic assortm...
2,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03,2015,1,4,"wow! yummy, different, delicious. our favorite..."
3,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,1,0,1,Cute interior and owner (?) gave us tour of up...,2017-01-14 20:54:15,2017,1,14,cute interior and owner ? gave us tour of upco...
4,JrIxlS1TzJ-iCu79ul40cQ,eUta8W_HdHMXPzLBBZhL1A,04UD14gamNjLY0IDYVhHJg,1,1,2,1,I am a long term frequent customer of this est...,2015-09-23 23:10:31,2015,9,23,i am a long term frequent customer of this est...


In [ ]:
review_df["date"] = pd.to_datetime(review_df["date"], errors="coerce")
review_df = review_df.dropna(subset=["date"])

review_df["year"] = review_df["date"].dt.year
review_df["month"] = review_df["date"].dt.month

In [ ]:
merge_df = review_df.merge(
     restaurant_df[["business_id","name", "address", "city","state", "postal_code", "categories", "stars","review_count", "is_open", ]],
    on="business_id",
    how="left",
    suffixes=("_review", "_business")
)

print("merged file: ", merge_df.shape)
merge_df.head()

merged file:  (750000, 22)


,review_id,user_id,business_id,stars_review,useful,funny,cool,text,date,year,...,text_cl,name,address,city,state,postal_code,categories,stars_business,review_count,is_open
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11,2018,...,"if you decide to eat here, just be aware it is...",Turning Point of North Wales,1460 Bethlehem Pike,North Wales,PA,19454,"Restaurants, Breakfast & Brunch, Food, Juice B...",3.0,169,1
1,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30,2014,...,family diner. had the buffet. eclectic assortm...,Kettle Restaurant,748 W Starr Pass Blvd,Tucson,AZ,85713,"Restaurants, Breakfast & Brunch",3.5,47,1
2,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03,2015,...,"wow! yummy, different, delicious. our favorite...",Zaika,2481 Grant Ave,Philadelphia,PA,19114,"Halal, Pakistani, Restaurants, Indian",4.0,181,1
3,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,1,0,1,Cute interior and owner (?) gave us tour of up...,2017-01-14 20:54:15,2017,...,cute interior and owner ? gave us tour of upco...,Melt,2549 Banks St,New Orleans,LA,70119,"Sandwiches, Beer, Wine & Spirits, Bars, Food, ...",4.0,32,0
4,JrIxlS1TzJ-iCu79ul40cQ,eUta8W_HdHMXPzLBBZhL1A,04UD14gamNjLY0IDYVhHJg,1,1,2,1,I am a long term frequent customer of this est...,2015-09-23 23:10:31,2015,...,i am a long term frequent customer of this est...,Dmitri's,795 S 3rd St,Philadelphia,PA,19147,"Mediterranean, Restaurants, Seafood, Greek",4.0,273,0


In [ ]:
merge_save = f"{folder}/merge_yelp_750k.csv"
merge_df.to_csv(merge_save, index=False)
print("Saved: ", merge_save)

Saved:  /content/drive/MyDrive/YELP/merge_yelp_750k.csv


In [ ]:
top_id = restaurant_df.sort_values("review_count", ascending=False).head(500)["business_id"]
app_df = merge_df[merge_df["business_id"].isin(set(top_id))].copy()

app_df = app_df.sort_values("date").groupby("business_id").tail(50)

In [ ]:
app_path = f"{folder}/APP_DATA.csv"
app_df.to_csv(app_path, index=False)
print("Saved App Data: ", app_path, "shape: ", app_df.shape)

Saved App Data:  /content/drive/MyDrive/YELP/APP_DATA.csv shape:  (5500, 22)
